In [2]:
import mimetypes
import os
import json
from dotenv import load_dotenv
from unstructured_client import UnstructuredClient
from unstructured_client.models.operations import CreateJobRequest
from unstructured_client.models.shared import BodyCreateJob, InputFiles

load_dotenv()

client = UnstructuredClient(
    api_key_auth=os.getenv("UNSTRUCTURED_API_KEY"),
    server_url=os.getenv("UNSTRUCTURED_API_URL")
)

print("Client connected")

Client connected


In [7]:
import json
import mimetypes
import os
import time
from dotenv import load_dotenv
from unstructured_client import UnstructuredClient
from unstructured_client.models.operations import CreateJobRequest, DownloadJobOutputRequest
from unstructured_client.models.shared import BodyCreateJob, InputFiles

load_dotenv()

INPUT_DIR = "../data/raw/building_and_construction/pcbus_working_together"
OUTPUT_DIR = "../data/processed/pcbus_working_together"

client = UnstructuredClient(
    api_key_auth=os.getenv("UNSTRUCTURED_API_KEY"),
    server_url="https://platform-api.transform.unstructured.io"
)

# Step 1: Create the job
input_files = []
for filename in os.listdir(INPUT_DIR):
    full_path = os.path.join(INPUT_DIR, filename)
    if not os.path.isfile(full_path) or not filename.endswith(".pdf"):
        continue
    content_type, _ = mimetypes.guess_type(full_path)
    input_files.append(
        InputFiles(
            content=open(full_path, "rb"),
            file_name=filename,
            content_type=content_type or "application/octet-stream"
        )
    )

print(f"Submitting {len(input_files)} file(s)...")

response = client.jobs.create_job(
    request=CreateJobRequest(
        body_create_job=BodyCreateJob(
            request_data=json.dumps({
                "job_nodes": [
                    {
                        "name": "Partitioner",
                        "type": "partition",
                        "subtype": "vlm",
                        "settings": {
                            "is_dynamic": True,
                            "allow_fast": True
                        }
                    }
                ]
            }),
            input_files=input_files
        )
    )
)

job_id = response.job_information.id
print(f"Job ID: {job_id}")

# Step 2: Poll until done
while True:
    response = client.jobs.get_job(request={"job_id": job_id})
    job_info = response.job_information
    status = job_info.status
    print(f"Job status: {status.value}")
    if status == "COMPLETED":
        print("Job completed.")
        break
    elif status in ("FAILED", "STOPPED"):
        raise RuntimeError(f"Job failed: {status}")
    time.sleep(10)

output_node_file_ids = [f.file_id for f in (job_info.output_node_files or [])]

# Step 3: Download results
os.makedirs(OUTPUT_DIR, exist_ok=True)
for file_id in output_node_file_ids:
    response = client.jobs.download_job_output(
        request=DownloadJobOutputRequest(job_id=job_id, file_id=file_id)
    )
    output_path = os.path.join(OUTPUT_DIR, f"{file_id}.json")
    with open(output_path, "w") as f:
        json.dump(response.any, f, indent=4)
    print(f"Saved: {output_path}")

Submitting 1 file(s)...


INFO: HTTP Request: POST https://platform-api.transform.unstructured.io/api/v1/jobs/ "HTTP/1.1 200 OK"


Job ID: a93271ce-d3ac-43c7-beee-66e177cb4b4e


INFO: HTTP Request: GET https://platform-api.transform.unstructured.io/api/v1/jobs/a93271ce-d3ac-43c7-beee-66e177cb4b4e "HTTP/1.1 200 OK"


Job status: SCHEDULED


...

Job status: IN_PROGRESS


INFO: HTTP Request: GET https://platform-api.transform.unstructured.io/api/v1/jobs/a93271ce-d3ac-43c7-beee-66e177cb4b4e "HTTP/1.1 200 OK"


Job status: COMPLETED
Job completed.


INFO: HTTP Request: GET https://platform-api.transform.unstructured.io/api/v1/jobs/a93271ce-d3ac-43c7-beee-66e177cb4b4e/download?file_id=PCBUs-Working-Together-GPG-7fcb7c71.pdf "HTTP/1.1 200 OK"


Saved: ../data/processed/pcbus_working_together/PCBUs-Working-Together-GPG-7fcb7c71.pdf.json


In [6]:
print(os.getenv("UNSTRUCTURED_API_URL"))


https://platform-api.transform.unstructured.io/api/v1


In [4]:
os.getcwd()

'/Users/rupertguppy/Desktop/R&D-Project/Health-Safety-AI/notebooks'

In [8]:
import json

output_files = os.listdir(OUTPUT_DIR)
print(f"Output files: {output_files}")

# Load the first output file
with open(os.path.join(OUTPUT_DIR, output_files[0]), "r") as f:
    elements = json.load(f)

print(f"Total elements: {len(elements)}")
print()

# Show first 10 elements
for el in elements[:10]:
    print(f"Type: {el['type']:20s} | Text: {el['text'][:100]}")

Output files: ['PCBUs-Working-Together-GPG-7fcb7c71.pdf.json']
Total elements: 684

Type: UncategorizedText    | Text: 
Type: Header               | Text: New Zealand Government WORKSAFE Mahi Haumaru Aotearoa
Type: Title                | Text: PCBUs working together
Type: Title                | Text: ADVICE WHEN CONTRACTING
Type: NarrativeText        | Text: June 2019
Type: UncategorizedText    | Text: 
Type: UncategorizedText    | Text: GOOD PRACTICE GUIDELINES
Type: NarrativeText        | Text: These guidelines are for Persons Conducting
Type: NarrativeText        | Text: a Business or Undertaking (PCBUs) who are
Type: NarrativeText        | Text: sharing a workplace with other businesses,


In [9]:
# Check for the word "different" - was broken before
for el in elements:
    if "different" in el["text"].lower():
        print(f"Type: {el['type']:20s} | Text: {el['text'][:120]}")
        break

print()

# Check what element types exist
from collections import Counter
type_counts = Counter(el["type"] for el in elements)
for t, count in type_counts.most_common():
    print(f"{t:25s}: {count}")

print()

# Check if any Table elements came through
for el in elements:
    if el["type"] == "Table":
        print(f"TABLE FOUND: {el['text'][:200]}")
        break

Type: NarrativeText        | Text: This is where several different contractors are working in the same place, such as a construction site, shopping centre 

NarrativeText            : 321
Title                    : 116
ListItem                 : 111
UncategorizedText        : 56
Header                   : 25
Footer                   : 21
Table                    : 13
PageNumber               : 11
Image                    : 9
Form                     : 1

TABLE FOUND: WHAT IS THE DUTY? WHAT IS THE REQUIREMENT? Primary duty You must ensure, so far as is reasonably practicable, that the of PCBUs health and safety of workers and other people are not put Section 36 at 


In [13]:
# Filter to only useful elements
keep_types = {"NarrativeText", "Title", "ListItem", "Table"}

clean_text = []
for el in elements:
    if el["type"] in keep_types:
        clean_text.append(el["text"])

# This is your clean text ready for chunking
full_text = "\n\n".join(clean_text)
print(full_text[:5000])

PCBUs working together

ADVICE WHEN CONTRACTING

June 2019

These guidelines are for Persons Conducting

a Business or Undertaking (PCBUs) who are

sharing a workplace with other businesses,

or are working as part of a contracting chain.

They provide advice on how you can meet

your duties under the Health and Safety at

Work Act 2015 (HSWA), illustrate di(erent

and provide examples of ways you can build

health and safety into contract management.

ACKNOWLEDGEMENTS

WorkSafe would like to acknowledge and thank the stakeholders

PCBUs working together

KEY POINTS

You must consult, cooperate and coordinate with

other PCBUs when working in a shared workplace,

or as part of a contracting chain

You can’t contract out of health and safety duties

You should always build health and safety into

contract management

CONTENTS

1.0 Understanding your duties under HSWA

1.1 What this guide is about

1.2 What you need to know before reading this guide

1.3 What WorkSafe expects in contract

In [14]:
broken = []
for el in elements:
    if "di(" in el["text"] or "e0" in el["text"] or "aﬀ" in el["text"] or "ﬁ" in el["text"] or "ﬂ" in el["text"]:
        broken.append(el["text"][:100])

print(f"Elements with encoding issues: {len(broken)}")
for b in broken[:5]:
    print(f"  {b}")

Elements with encoding issues: 90
  Work Act 2015 (HSWA), illustrate di(erent
  Appendix 1: Pre-qualiﬁcation questionnaire example Appendix 2: Information for tenderer form example
  1 2 What WorkSafe expects from PCBUs 3 Ability to inﬂuence and control 4 Key considerations when cho
  ﬁgures
  Where duties are shared, all PCBUs have a responsibility to meet those duties, to the extent that th


In [17]:
def clean_text(text):
    replacements = {
        "ﬁ": "fi",
        "ﬂ": "fl",
        "ﬀ": "ff",
        "aﬀ": "aff",
        "eﬀ": "eff",
        "di(erent": "different",
        "e0": "eff",
        "o0": "off",
        # Add these to your replacements
        "(section ,6": "(section 16",
        "(section ,9": "(section 19",
        "(section .-": "(section 20",
        "(section ..": "(section 22",
        "(section .5": "(section 25",

    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

# Test it
keep_types = {"NarrativeText", "Title", "ListItem", "Table"}

clean_elements = []
for el in elements:
    if el["type"] in keep_types:
        clean_elements.append(clean_text(el["text"]))

# Check if issues are fixed
broken_count = sum(1 for t in clean_elements if "di(" in t or "ﬁ" in t or "ﬂ" in t)
print(f"Remaining issues: {broken_count}")
print()
print("\n\n".join(clean_elements[:10]))

Remaining issues: 0

PCBUs working together

ADVICE WHEN CONTRACTING

June 2019

These guidelines are for Persons Conducting

a Business or Undertaking (PCBUs) who are

sharing a workplace with other businesses,

or are working as part of a contracting chain.

They provide advice on how you can meet

your duties under the Health and Safety at

Work Act 2015 (HSWA), illustrate different


In [18]:
for i, el in enumerate(elements):
    if el["type"] == "Table":
        print(f"--- Table {i} ---")
        print(clean_text(el["text"][:500]))
        print()

--- Table 111 ---
WHAT IS THE DUTY? WHAT IS THE REQUIREMENT? Primary duty You must ensure, so far as is reasonably practicable, that the of PCBUs health and safety of workers and other people are not put Section 36 at risk by your work. This is called the ‘primary duty of care’. Health and Safety This means ensuring, so far as is reasonably practicable: at Work Act 2015 – the health and safety of workers while they are at work (including contractors, subcontractors and their workers) – the health and safety of wo

--- Table 127 ---
Lead PCBU – To be a health and safety leader Example: – To set clear health and safety expectations and incorporate these into contracts with The owners of a contractors large mall that is – To work with designers to eliminate risks so far as is reasonably practicable, or minimise risks being refurbished if they cannot be eliminated – To choose the best contractors and site managers for the job using prequalification, not simply choosing them based on cost –

In [19]:
# Load the JSON
import json
import re

with open(os.path.join(OUTPUT_DIR, output_files[0]), "r") as f:
    elements = json.load(f)

# Cleaning function with all fixes
def clean_text(text):
    replacements = {
        "ﬁ": "fi",
        "ﬂ": "fl",
        "ﬀ": "ff",
        "aﬀ": "aff",
        "eﬀ": "eff",
        "di(erent": "different",
        "e0": "eff",
        "o0": "off",
        "(section ,6": "(section 16",
        "(section ,9": "(section 19",
        "(section .-": "(section 20",
        "(section ..": "(section 22",
        "(section .5": "(section 25",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    # Strip checkbox artifacts (stray 2 before capital letters)
    text = re.sub(r'(?<!\d)2(?=[A-Z])', '', text)
    return text

# Filter and clean
keep_types = {"NarrativeText", "Title", "ListItem", "Table"}
cleaned = [clean_text(el["text"]) for el in elements if el["type"] in keep_types]

# --- ISSUE SCAN ---
print("=== ISSUE SCAN ===")
issues_found = 0
patterns = {
    "Broken 'different'": "di(erent",
    "Broken ligature fi": "ﬁ",
    "Broken ligature fl": "ﬂ",
    "Broken ligature ff": "ﬀ",
    "Broken e0": "e0c",
    "Broken o0": "o0c",
    "Section number ,": "(section ,",
    "Section number .": "(section .",
    "Checkbox artifact": None,  # handled separately
}

for name, pattern in patterns.items():
    if pattern:
        count = sum(1 for t in cleaned if pattern in t)
    else:
        count = sum(1 for t in cleaned if re.search(r'(?<!\d)2(?=[A-Z])', t))
    if count > 0:
        issues_found += count
        print(f"  FOUND: {name} ({count} occurrences)")

if issues_found == 0:
    print("  No known issues found.")

# Check for any non-ASCII characters that might be new problems
print("\n=== UNUSUAL CHARACTERS ===")
unusual = set()
for t in cleaned:
    for char in t:
        if ord(char) > 127 and char not in "–—''""•…àáâãäåèéêëìíîïòóôõöùúûüñçÀÁÂÃÄÅÈÉÊËÌÍÎÏÒÓÔÕÖÙÚÛÜÑÇ":
            unusual.add((char, hex(ord(char))))

if unusual:
    for char, code in sorted(unusual, key=lambda x: x[1]):
        examples = [t[:80] for t in cleaned if char in t][:2]
        print(f"  Char: '{char}' ({code})")
        for ex in examples:
            print(f"    Example: {ex}")
else:
    print("  None found.")

# --- OUTPUT FULL TEXT ---
full_text = "\n\n".join(cleaned)
print(f"\n=== STATS ===")
print(f"Total elements kept: {len(cleaned)}")
print(f"Total characters: {len(full_text)}")

# Save to file
output_path = os.path.join(OUTPUT_DIR, "PCBUs-Working-Together-CLEAN.txt")
with open(output_path, "w") as f:
    f.write(full_text)
print(f"Saved to: {output_path}")

# Preview
print(f"\n=== PREVIEW (first 1000 chars) ===")
print(full_text[:1000])

=== ISSUE SCAN ===
  FOUND: Section number , (1 occurrences)

=== UNUSUAL CHARACTERS ===
  Char: '‘' (0x2018)
    Example: 1 PCBU is a ‘person conducting a business or undertaking’. They may be an indivi
    Example: WHAT IS THE DUTY? WHAT IS THE REQUIREMENT? Primary duty You must ensure, so far 
  Char: '’' (0x2019)
    Example: You can’t contract out of health and safety duties
    Example: 1 PCBU is a ‘person conducting a business or undertaking’. They may be an indivi

=== STATS ===
Total elements kept: 561
Total characters: 60820
Saved to: ../data/processed/pcbus_working_together/PCBUs-Working-Together-CLEAN.txt

=== PREVIEW (first 1000 chars) ===
PCBUs working together

ADVICE WHEN CONTRACTING

June 2019

These guidelines are for Persons Conducting

a Business or Undertaking (PCBUs) who are

sharing a workplace with other businesses,

or are working as part of a contracting chain.

They provide advice on how you can meet

your duties under the Health and Safety at

Work Act 2015 

In [20]:
for t in cleaned:
    if "(section ," in t:
        print(t[:150])

TERM LEGAL DEFINITION (AS NOTED) OR BRIEF EXPLANATION Contractor A PCBU who carries out temporary, work under contract. Control measure Is a way of el
